In [1]:
# Kill all processes on the GPU
!fuser -v /dev/nvidia* -k

                     USER        PID ACCESS COMMAND
/dev/nvidia0:        root       6091 F...m python3
                     root       6681 F.... wandb-xpu
/dev/nvidiactl:      root       6091 F...m python3
                     root       6681 F.... wandb-xpu
/dev/nvidia-uvm:     root       6091 F...m python3
                     root       6681 F.... wandb-xpu


In [2]:
# Check the GPU status
!nvidia-smi

Thu Sep 17 12:04:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   75C    P0             36W /   70W |   13053MiB /  15360MiB |     41%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Libraries

In [3]:
%%capture
!uv pip uninstall torchao torchaudio torchvision -y
!uv pip install \
    "transformers==4.53.3" \
    "peft==0.17.1" \
    "trl" \
    "accelerate" \
    "bitsandbytes" \
    "wandb"

In [4]:
import os
import math
import torch
from datetime import datetime
from transformers import (
    AutoModelForMaskedLM, 
    AutoModelForQuestionAnswering,
    DataCollatorForLanguageModeling, 
    DataCollatorWithPadding,
    AutoTokenizer,
    Trainer, 
    TrainingArguments,
    EarlyStoppingCallback,
)
from peft import (
    LoraConfig, 
    get_peft_model,
    PeftModel,
)
from datasets import load_dataset, Dataset

# Utilities

In [5]:
def load_train_val_datasets(
    lang, # e.g., 'en' | 'ja' | 'id'
    task, # 'wikipedia' | 'squad'
    train_size, val_size,
):
    # Validate that if the task is 'squad', the language must be 'en'
    if task == 'squad':
        assert lang == 'en', "SQuAD is English-only."
    
    # Define dataset configurations for each task
    data_configs = {
        'wikipedia': {
            'data_id': 'wikimedia/wikipedia',
            'data_dir': f'20231101.{lang}',
            'train_split': 'train',
            'val_split': 'train',
        },
        'squad': {
            'data_id': 'rajpurkar/squad',
            'data_dir': None,
            'train_split': 'train',
            'val_split': 'validation',
        },
    }
    
    # Validate that the specified task is supported
    assert task in data_configs, (
        f"Unsupported task: {task}. "
        f"Supported tasks: {list(data_configs.keys())}"
    )

    # Set up Hugging Face dataset configuration
    data_id = data_configs[task]['data_id']
    data_dir = data_configs[task]['data_dir']
    train_split = data_configs[task]['train_split']
    val_split = data_configs[task]['val_split']

    if train_split == val_split:
        # If the train and validation splits are the same, we need to sample from the same dataset stream
        dataset_stream = load_dataset(
            data_id,
            data_dir=data_dir,
            split=train_split,
            streaming=True,
        )

        train_data = []
        val_data = []

        for i, example in enumerate(dataset_stream):
            if i < train_size:
                train_data.append(example)
            elif i < train_size + val_size:
                val_data.append(example)
            else:
                break

    else:
        # If the train and validation splits are different, we can sample from each split separately
        def sample_split(split, size):
            dataset_stream = load_dataset(
                data_id,
                data_dir=data_dir,
                split=split,
                streaming=True,
            )

            data = []
            for i, example in enumerate(dataset_stream):
                if i >= size:
                    break
                data.append(example)
            return data

        train_data = sample_split(train_split, train_size)
        val_data = sample_split(val_split, val_size)

    return (
        Dataset.from_list(train_data),
        Dataset.from_list(val_data),
    )

# Configuration

In [6]:
# Run configuration
SEED = 42
USERNAME = 'alxxtexxr'
TASK = 'wikipedia'  # 'wikipedia' | 'squad'
LANG = 'en'  # wikipedia: 'en' | '<any_lang_code>', squad: 'en'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Model configuration
MODEL_ID = 'FacebookAI/xlm-roberta-base'
MODEL_NAME = 'XLM-R-Base'
# -------- New training --------
# RESUME_MODEL_ID = None
# RESUME_CKPT_STEP = None
# -------- Resume training --------
RESUME_MODEL_ID = 'alxxtexxr/XLM-R-Base-wikipedia-en-10K-LoRA-v260917095723'
RESUME_CKPT_STEP = 800

# LoRA configuration
LORA_RANK = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.1
LORA_TARGET_MODULES = [
    # Not 'all-linear', since we exclude 'qa_outputs' for question answering tasks
    'query', 
    'key', 
    'value', 
    # Not all 'dense' layers, since we exclude 'lm_head.dense' for masked language modeling
    'intermediate.dense',
    'output.dense',
]

# Data configuration
# -------- 1K (89%:11%) --------
# TRAIN_SIZE = 1000
# VAL_SIZE = 125
# -------- 5K (89%:11%) --------
# TRAIN_SIZE = 5000
# VAL_SIZE = 625
# -------- 10K (80%:20%) --------
TRAIN_SIZE = 10_000
VAL_SIZE = 2_500

# Training configuration
MINI_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
LR = 2e-4
MLM_PROB = 0.15
# -------- D: 1K | 5K, E: 20 --------
# NUM_EPOCHS = 20
# WARMUP_STEPS = 50
# EVAL_STEPS = 20
# SAVE_STEPS = 20
# LOGGING_STEPS = 10
# -------- D: 10K, E: 20 --------
# NUM_EPOCHS = 20
# WARMUP_STEPS = 50
# EVAL_STEPS = 200
# SAVE_STEPS = 200
# LOGGING_STEPS = 50
# -------- D: 10K, E: 3 --------
NUM_EPOCHS = 3
WARMUP_STEPS = 100
EVAL_STEPS = 200
SAVE_STEPS = 200
LOGGING_STEPS = 50

In [7]:
# Resume training configuration
resume_from_checkpoint = bool(RESUME_MODEL_ID)
if resume_from_checkpoint:
    model_name = RESUME_MODEL_ID
    run_name = model_name.split('/')[-1]
    hub_model_id = RESUME_MODEL_ID
    
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=hub_model_id, local_dir=model_name)
    
    if RESUME_CKPT_STEP:
        resume_from_checkpoint = f"{hub_model_id}/checkpoint-{RESUME_CKPT_STEP}"
        # Ensure the checkpoint exists
        assert os.path.exists(resume_from_checkpoint), f"Checkpoint {resume_from_checkpoint} does not exist."
            
else:
    run_name = f'{MODEL_NAME}-{TASK}-{LANG}-{TRAIN_SIZE/1000:g}K-LoRA-v{datetime.now().strftime("%y%m%d%H%M%S")}'
    hub_model_id = f'{USERNAME}/{run_name}'
base_hub_model_id, version = hub_model_id.rsplit('-v', 1)
hub_merged_model_id = f'{base_hub_model_id}-Merged-v{version}'

print("Resume from checkpoint:", resume_from_checkpoint)
print("Model name:", MODEL_NAME)
print("Run name:", run_name)
print("Hub model ID:", hub_model_id)
print("Hub merged model ID:", hub_merged_model_id)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 53 files:   0%|          | 0/53 [00:00<?, ?it/s]

checkpoint-200/rng_state.pth:   0%|          | 0.00/14.6k [00:00<?, ?B/s]

checkpoint-200/optimizer.pt:   0%|          | 0.00/5.56M [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/10.6M [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

trainer_state.json: 0.00B [00:00, ?B/s]

runs/Sep17_09-59-13_a8d8b22e0e8a/events.(…):   0%|          | 0.00/9.62k [00:00<?, ?B/s]

training_args.bin:   0%|          | 0.00/5.91k [00:00<?, ?B/s]

Resume from checkpoint: alxxtexxr/XLM-R-Base-wikipedia-en-10K-LoRA-v260917095723/checkpoint-800
Model name: XLM-R-Base
Run name: XLM-R-Base-wikipedia-en-10K-LoRA-v260917095723
Hub model ID: alxxtexxr/XLM-R-Base-wikipedia-en-10K-LoRA-v260917095723
Hub merged model ID: alxxtexxr/XLM-R-Base-wikipedia-en-10K-LoRA-Merged-v260917095723


In [8]:
# Set environment variables for wandb logging
os.environ['WANDB_PROJECT'] = 'legamex'
os.environ['WANDB_NAME'] = run_name
# os.environ['WANDB_LOG_MODEL'] = 'checkpoint' # Control whether checkpoints get uploaded to wandb as artifacts

# Model

In [9]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Determine the appropriate model classes based on the task
if TASK == 'squad':
    model_cls = AutoModelForQuestionAnswering
    task_type = 'QUESTION_ANS'
    data_collator = DataCollatorWithPadding(tokenizer)
    label_names = ['start_positions', 'end_positions']
else:
    model_cls = AutoModelForMaskedLM
    task_type = 'FEATURE_EXTRACTION'
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=True,
        mlm_probability=MLM_PROB,
    )
    label_names = ['labels']

# Load the model
model = model_cls.from_pretrained(MODEL_ID)

Some weights of the model checkpoint at FacebookAI/xlm-roberta-base were not used when initializing XLMRobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [10]:
# Sanity check
# for n, p in model.named_parameters():
#     print(n, p.requires_grad)

In [11]:
if resume_from_checkpoint:
    # Load the LoRA adapter from the checkpoint and ensure it's in training mode
    model = PeftModel.from_pretrained(model, resume_from_checkpoint)
    model.inference_mode = False  # Disable inference-only flag
    model.enable_adapter_layers() # Explicitly unfreeze LoRA weights
else:
    # Set up a LoRA configuration and apply it to the model
    lora_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias='none',
        task_type=task_type,
        target_modules=LORA_TARGET_MODULES,
    )
    model = get_peft_model(model, lora_config)
model = model.to(DEVICE)

model.print_trainable_parameters()
print("device:", model.device)

trainable params: 2,654,208 || all params: 280,949,394 || trainable%: 0.9447
device: cuda:0


In [ ]:
# Sanity check
# for n, p in model.named_parameters():
#     if p.requires_grad:
#         print(n, p.requires_grad)

# Data

In [13]:
# Load the dataset
train_dataset, val_dataset = load_train_val_datasets(lang=LANG, task=TASK, 
                                                     train_size=TRAIN_SIZE, val_size=VAL_SIZE)

print("Train dataset:")
print(train_dataset)
print()
print("Validation dataset:")
print(val_dataset)

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Train dataset:
Dataset({
    features: ['id', 'url', 'title', 'text'],
    num_rows: 10000
})

Validation dataset:
Dataset({
    features: ['id', 'url', 'title', 'text'],
    num_rows: 2500
})


In [14]:
# Preprocess the dataset
if TASK == 'squad':
    def preprocess_squad(examples):
        # Tokenize question + context with offset mapping
        tokenized = tokenizer(
            examples['question'],
            examples['context'],
            truncation='only_second',
            max_length=384,
            # stride=128,
            return_offsets_mapping=True,
            # padding='max_length',
        )

        # Prepare label lists
        start_positions = []
        end_positions = []

        for i, offsets in enumerate(tokenized['offset_mapping']):
            # Find which tokens belong to the context (not the question, not special tokens)
            sequence_ids = tokenized.sequence_ids(i)

            # SQuAD v1.1 always has exactly one answer; take the first
            answer = examples['answers'][i]
            answer_start_char = answer['answer_start'][0]
            answer_text = answer['text'][0]
            answer_end_char = answer_start_char + len(answer_text)

            # Locate the token span that corresponds to the answer
            token_start = None
            token_end = None
            for idx, (offset_start, offset_end) in enumerate(offsets):
                # Ignore question tokens and special tokens
                if sequence_ids[idx] != 1:
                    continue
                # Token fully inside answer
                if offset_start >= answer_start_char and offset_end <= answer_end_char:
                    if token_start is None:
                        token_start = idx
                    token_end = idx
                # Token partially overlapping (shouldn’t happen with whitespace splits)
                elif offset_start < answer_end_char and offset_end > answer_start_char:
                    if token_start is None:
                        token_start = idx
                    token_end = idx

            # If answer is out of bounds (truncated), set to CLS token index
            if token_start is None or token_end is None:
                token_start = 0
                token_end = 0

            start_positions.append(token_start)
            end_positions.append(token_end)

        tokenized['start_positions'] = start_positions
        tokenized['end_positions'] = end_positions

        return tokenized
    
    train_dataset = train_dataset.map(preprocess_squad, batched=True, remove_columns=train_dataset.column_names)
    val_dataset = val_dataset.map(preprocess_squad, batched=True, remove_columns=val_dataset.column_names)
else:
    def tokenize_text(examples):
        tokenized = tokenizer(
            examples['text'],
            max_length=512,
            truncation=True,
            padding='max_length',
        )
        tokenized['labels'] = tokenized['input_ids'].copy()
        return tokenized

    train_dataset = train_dataset.map(tokenize_text, batched=True, remove_columns=train_dataset.column_names)
    val_dataset = val_dataset.map(tokenize_text, batched=True, remove_columns=val_dataset.column_names)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

# Training

In [15]:
# Calculate the maximum number of training steps
max_steps = math.ceil(len(train_dataset) / (MINI_BATCH_SIZE * GRAD_ACCUM_STEPS)) * NUM_EPOCHS
print("Calculated max steps:", max_steps)

# Set up a trainer
training_args = TrainingArguments(
    # Training arguments
    seed=SEED,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    per_device_train_batch_size=MINI_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    max_steps = max_steps,
    warmup_steps = WARMUP_STEPS,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    optim='adamw_8bit',
    max_grad_norm=1.0,
    weight_decay=0.01,
    
    # Validation arguments
    eval_strategy='steps',
    eval_steps=EVAL_STEPS,
    
    # Logging arguments
    logging_strategy='steps',
    logging_steps=LOGGING_STEPS,
    # logging_first_step=True,
    report_to=['tensorboard', 'wandb'],
    
    # Saving arguments
    save_strategy='steps',
    save_steps=SAVE_STEPS,
    # save_total_limit=5, # 1 best + 4 recent checkpoints. WARN: It doesn't work
    
    # With load_best_model_at_end=True, your save_strategy will be ignored and default to eval_strategy.
    # So you will find one checkpoint at the end of each epoch.
    # https://discuss.huggingface.co/t/trainer-not-saving-after-save-steps/5464
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better = False,

    # run_name=run_name,
    output_dir=run_name,
    hub_model_id=hub_model_id,
    push_to_hub=True,
    hub_strategy='all_checkpoints',
    hub_always_push=True,
)
trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    args=training_args,
    # label_names=['labels'],
    callbacks = [
        EarlyStoppingCallback(
            early_stopping_patience=3,
            # early_stopping_threshold = 0.001,
        )
    ],
)
trainer.label_names = label_names

Calculated max steps: 1875


No label_names provided for model class `PeftModelForFeatureExtraction`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [16]:
# Start training
trainer_stats = trainer.train(resume_from_checkpoint=resume_from_checkpoint)

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: alimtegar to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss
1000,1.504800,1.439549
1200,1.468400,1.450012
1400,1.463200,1.460850
1600,1.481200,1.449526


# Merging

In [17]:
# If the task is SQuAD, upload the merged model and tokenizer
if TASK == 'squad':
    # After the training finishes, merge the LoRA into the base model and save everything
    model = model.eval() # Good practice
    merged_model = model.merge_and_unload()

    # Upload the merged model to Hugging Face
    merged_model.push_to_hub(hub_merged_model_id)
    tokenizer.push_to_hub(hub_merged_model_id)

    print(f"Merged model uploaded to: https://huggingface.co/{hub_merged_model_id}")